# 01 - Exploratory Data Analysis

Load and explore NYC TLC trip data to understand demand patterns.

In [ ]:
import sys; sys.path.insert(0, '..')
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.download import download_tlc_data, download_zone_lookup, read_trip_parquet
from src.data.preprocess import filter_trips, aggregate_hourly_demand

In [ ]:
raw_dir = Path('../data/raw')
ext_dir = Path('../data/external')
download_tlc_data([2024], [1], raw_dir)
download_zone_lookup(ext_dir)

In [ ]:
df = read_trip_parquet(raw_dir / 'yellow_tripdata_2024-01.parquet')
df.head()

In [ ]:
df_filtered = filter_trips(df)
hourly = aggregate_hourly_demand(df_filtered)
hourly.describe()

In [ ]:
zones = pl.read_csv(ext_dir / 'taxi_zone_lookup.csv')
zone_counts = hourly.group_by('PULocationID').agg(pl.sum('demand')).join(zones, left_on='PULocationID', right_on='LocationID').sort('demand', descending=True)
zone_counts.head(10).plot.bar(x='Zone', y='demand')
plt.title('Top 10 Pickup Zones by Total Demand')
plt.xticks(rotation=45)
plt.show()

In [ ]:
hourly_daily = hourly.group_by(pl.col('pickup_hour').dt.date()).agg(pl.sum('demand')).sort('pickup_hour')
plt.figure(figsize=(14, 4))
plt.plot(hourly_daily['pickup_hour'], hourly_daily['demand'])
plt.title('Daily Total Demand — January 2024')
plt.ylabel('Trips')
plt.show()